# 🍽️ Augmentation Dataset — Mets Africains v6
## 5 générations SD (modifications réelles) + 5 transformations PIL (environnement)

**Inspiré de l'image de référence :** même plat de base, 5 variations réalistes et cohérentes.

**Pipeline :**
- **5 SD Inpainting** : modifications sémantiques réelles du contenu
  - Remplacement de protéine (poisson → viande braisée, poulet → pintade...)
  - Changement de sauce ou volume
  - Ajout d'accompagnement naturel (alloco, riz, légumes)
  - Changement de contenant (calebasse, canari, feuille bananier)
- **5 PIL** : variations d'environnement/photographie
  - Luminosité, couleur, angle, éclairage — sans toucher au contenu

**⚠️ OBLIGATOIRE :** Exécution → Modifier le type → **GPU T4**

**Structure ZIP attendue :**
```
dataset.zip
├── annotations.xlsx
└── images/
    ├── foutou_009.jpg
    └── ...
```

In [ ]:
!pip install diffusers transformers accelerate xformers torch Pillow pandas openpyxl -q
import torch, os, zipfile, unicodedata, glob as _glob
from pathlib import Path
import pandas as pd
from PIL import Image, ImageEnhance, ImageOps, ImageFilter, ImageDraw
import numpy as np
print(f'✅ GPU : {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else '❌ Active GPU T4 : Exécution > Modifier le type')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.3 MB/s eta 0:00:00
✅ GPU : Tesla T4


In [ ]:
from google.colab import files
print('📁 Upload ton ZIP (annotations.xlsx + dossier images/)')
uploaded = files.upload()
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f'✅ Extrait : {fname}')
    elif fname.endswith('.xlsx'):
        print(f'✅ Excel : {fname}')
import subprocess; subprocess.run(['ls', '-la'])

📁 Upload ton ZIP (annotations.xlsx + dossier images/)


Saving Colabfiles.zip to Colabfiles.zip
✅ Extrait : Colabfiles.zip


CompletedProcess(args=['ls', '-la'], returncode=0)

In [ ]:
# ════════════════════════════════════════
# ⚙️  CONFIG — adapter ces 4 lignes
# ════════════════════════════════════════
EXCEL_INPUT   = 'Colabfiles/image_site.xlsx'
SHEET_INDEX   = 0          # 0=feuille1, 1=feuille2
IMAGE_COL     = 'image_id'
IMAGES_FOLDER = 'Colabfiles/images/'

OUTPUT_FOLDER = 'images_augmentees/'
EXCEL_OUTPUT  = 'annotations_augmentees.xlsx'
SD_MODEL      = 'stable-diffusion-v1-5/stable-diffusion-inpainting'
IMG_SIZE      = 512
NUM_STEPS     = 40
GUIDANCE      = 11.0
STRENGTH      = 0.75   # 0.75 = modifie la zone masquée sans tout refaire

N_SD  = 5
N_PIL = 5

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'✅ Config OK — {N_SD} SD + {N_PIL} PIL = {N_SD+N_PIL} images/photo')

✅ Config OK — 5 SD + 5 PIL = 10 images/photo


In [ ]:
# ════════════════════════════════════════════════════════════════
# 📐   5 TRANSFORMATIONS PIL — COCKTAIL DE 4+ PROPRIÉTÉS
#
# Chaque variante applique désormais un mix complexe et réaliste :
# Spatial (Rotation/Zoom) + Luminosité/Contraste + Colorimétrie.
# ════════════════════════════════════════════════════════════════

import numpy as np
from PIL import Image, ImageEnhance, ImageOps

def bord(img):
    arr = np.array(img.convert('RGB'))
    b   = np.concatenate([arr[0], arr[-1], arr[:,0], arr[:,-1]])
    return tuple(b.mean(axis=0).astype(int))

def rot(img, a):
    return img.rotate(a, resample=Image.BICUBIC, expand=False, fillcolor=bord(img))

def rgb(img, r=0, g=0, b=0):
    ri, gi, bi = img.convert('RGB').split()
    ri = ri.point(lambda x: min(255, max(0, x+r)))
    gi = gi.point(lambda x: min(255, max(0, x+g)))
    bi = bi.point(lambda x: min(255, max(0, x+b)))
    return Image.merge('RGB', (ri, gi, bi))

def vign(img, f=80):
    w, h = img.size; cx, cy = w/2, h/2
    m = Image.new('L', (w,h))
    m.putdata([max(0, int(255-f*((x-cx)**2/cx**2+(y-cy)**2/cy**2)**0.5))
               for y in range(h) for x in range(w)])
    return Image.composite(img.convert('RGB'), Image.new('RGB',(w,h),(0,0,0)), m)

def spot(img, direction='gauche', force=0.38):
    w, h = img.size; px = []
    for y in range(h):
        for x in range(w):
            t = x/w if direction=='gauche' else 1-x/w if direction=='droite' else y/h if direction=='haut' else 1-y/h
            px.append(min(255, max(0, int(255*(1-force*(1-t))))))
    m = Image.new('L', (w,h)); m.putdata(px)
    return Image.composite(img.convert('RGB'), Image.new('RGB',(w,h),(0,0,0)), m)

def zoom(img, factor=1.12):
    """Zoom numérique : Recadre au centre et redimensionne à la taille d'origine."""
    w, h = img.size
    if factor == 1.0: return img
    nw, nh = int(w / factor), int(h / factor)
    left = (w - nw) // 2
    top = (h - nh) // 2
    return img.crop((left, top, left + nw, top + nh)).resize((w, h), Image.BICUBIC)

# Raccourcis PIL Enhance
lum = lambda i,f: ImageEnhance.Brightness(i).enhance(f)
sat = lambda i,f: ImageEnhance.Color(i).enhance(f)
ctr = lambda i,f: ImageEnhance.Contrast(i).enhance(f)

PIL_RECETTES = [
    # 1 — Lumière chaude dorée (Restaurant chic / Tamisé)
    # [Mix 4] : Zoom + Coloration chaude + Luminosité boostée + Saturation forte
    ('env_lumiere_chaude',
     lambda img: sat(lum(rgb(zoom(img, 1.10), r=35, g=12, b=-25), 1.15), 1.35)),

    # 2 — Ambiance flash contrastée (Photo rapprochée / Smartphone)
    # [Mix 4] : Légère rotation + Assombrissement + Contraste violent + Vignettage ciblé
    ('env_flash_contraste',
     lambda img: vign(ctr(lum(rot(img, -3), 0.80), 1.65), f=95)),

    # 3 — Vue extérieure grand angle (Tons froids de journée)
    # [Mix 5] : Rotation + Zoom (crop des bords noirs) + Teinte froide + Désaturation + Contraste
    ('env_exterieur_froid',
     lambda img: ctr(sat(rgb(zoom(rot(img, 6), 1.12), r=-20, g=-5, b=22), 0.85), 1.15)),

    # 4 — Changement d'angle drastique (Angle inversé et lumineux)
    # [Mix 5] : Miroir + Faisceau lumineux + Coloration + Zoom immersif + Ajustement expo
    ('env_miroir_lateral',
     lambda img: lum(zoom(rgb(spot(ImageOps.mirror(img), 'droite', 0.45), r=18, g=5, b=-15), 1.15), 0.95)),

    # 5 — Ambiance de nuit / Maquis (Chaud, sombre et intimiste)
    # [Mix 5] : Inclinaison + Forte coloration ambre + Sous-exposition + Gros vignettage + Saturation
    ('env_nuit_maquis',
     lambda img: sat(vign(lum(rgb(rot(img, -5), r=45, g=15, b=-35), 0.65), f=130), 1.25)),
]

assert len(PIL_RECETTES) == 5
print('✅ 5 transformations PIL multi-propriétés (4+ facteurs) prêtes :')
for i, (n, _) in enumerate(PIL_RECETTES, 1):
    print(f'   {i}. {n}')

✅ 5 transformations PIL multi-propriétés (4+ facteurs) prêtes :
   1. env_lumiere_chaude
   2. env_flash_contraste
   3. env_exterieur_froid
   4. env_miroir_lateral
   5. env_nuit_maquis


In [ ]:
MOTS_CLES = {
    'foutou':          ['foutou'],
    'alloco':           ['alloco'],
    'attieke':         ['attieke'],
    'kedjenou':        ['kedjenou'],
    'gombo':           ['gombo'],
    'thieboudienne':   ['thieboudienne', 'thiebu', 'ceebu'],
    'mafe':            ['mafe'],
    'yassa':           ['yassa'],
    'thiakry':         ['thiakry', 'thiakri'],
    'domoda':          ['domoda'],
}

def detecter_plat(nom_fichier):
    import unicodedata
    nom = nom_fichier.lower().replace('_', ' ').replace('-', ' ')
    nom = unicodedata.normalize('NFKD', nom)
    nom = ''.join(c for c in nom if not unicodedata.combining(c))
    for plat, mots in MOTS_CLES.items():
        if any(m in nom for m in mots):

            return 'kedjenou'
    return 'kedjenou'

SD_AJOUTS = {

    'foutou': [
        {
            'id': 'poisson_braise',
            'label': 'Poisson braisé entier à la place de la viande',
            'mask': 'center',
            'prompt': (
                'Professional food photography of a West African foutou dish. Modifying the central protein: '
                'replacing the meat with one whole golden-brown braised tilapia fish with crisp scored skin, '
                'resting naturally inside the rich orange palm nut sauce. The existing pounded plantain balls '
                'and overall composition remain perfectly integrated. Realistic textures, identical lighting.'
            ),
        },
        {
            'id': 'sauce_gombo',
            'label': 'Sauce gombo à la place de la sauce graine',
            'mask': 'center',
            'prompt': (
                'Professional food photography of a West African foutou dish. Modifying the sauce area: '
                'replacing the orange palm nut sauce with a thick, vibrant dark green okra soup (sauce gombo), '
                'viscous texture, with visible pieces of seafood and tender meat. The original plantain balls '
                'remain untouched, blending naturally with the new sauce.'
            ),
        },
        {
            'id': 'viande_boeuf',
            'label': 'Gros morceaux de boeuf braisé dans la sauce',
            'mask': 'center',
            'prompt': (
                'Professional food photography of a West African foutou dish. Adding large, succulent chunks '
                'of tender braised beef with a rich, glossy glaze, placed naturally inside the orange palm nut sauce '
                'next to the existing plantain balls. Blending seamlessly with the original plate, exact same lighting.'
            ),
        },
        {
            'id': 'calebasse',
            'label': 'Servi dans une calebasse africaine',
            'mask': 'border',
            'prompt': (
                'Professional food photography. Modifying the container: the existing West African foutou and '
                'sauce are now elegantly nested inside an authentic traditional African calabash gourd bowl, '
                'showing a natural, smooth beige-brown organic wood texture, replacing the original plate border seamlessly.'
            ),
        },
        {
            'id': 'escargots_afric',
            'label': 'Escargots africains dans la sauce',
            'mask': 'center',
            'prompt': (
                'Professional food photography of a West African foutou dish. Adding three large, perfectly cooked '
                'African land snails, glistening and integrated naturally into the rich orange palm nut sauce alongside '
                'the existing plantain balls. Clean composition, photorealistic.'
            ),
        },
    ],

    'alloco': [
    {
        'id': 'poisson_braise',
        'label': 'Poisson braisé entier en accompagnement',
        'mask': 'right_half',
        'prompt': (
            'Professional food photography of an authentic West African alloco dish. '
            'Adding one whole golden-brown braised tilapia with visible grill marks, '
            'naturally placed beside the fried ripe plantain slices. '
            'The existing alloco, onions, peppers and presentation remain unchanged. '
            'The fish blends naturally with the original meal, maintaining identical lighting, '
            'camera angle, colors and realistic textures.'
        ),
    },
    {
        'id': 'viande_braisee',
        'label': 'Viande de boeuf braisée en accompagnement',
        'mask': 'right_half',
        'prompt': (
            'Professional food photography of an authentic West African alloco dish. '
            'Adding several juicy pieces of braised beef with a rich caramelized glaze, '
            'served naturally beside the fried plantain. '
            'The original alloco and garnishes remain untouched. '
            'The new meat integrates seamlessly with the existing composition, '
            'keeping the same lighting, perspective and photorealistic appearance.'
        ),
    },
    {
        'id': 'omelette',
        'label': 'Omelette maison en accompagnement',
        'mask': 'right_half',
        'prompt': (
            'Professional food photography of an authentic West African alloco dish. '
            'Adding a freshly cooked golden folded omelette beside the fried plantain, '
            'with a soft texture and lightly browned surface. '
            'Keep the original alloco, onions and peppers exactly as they are. '
            'Natural presentation, identical lighting, framing and realistic food textures.'
        ),
    },
    {
        'id': 'brochettes_boeuf',
        'label': 'Brochettes de boeuf grillées',
        'mask': 'right_half',
        'prompt': (
            'Professional food photography of an authentic West African alloco dish. '
            'Adding two grilled beef skewers with lightly charred edges, '
            'served naturally next to the fried ripe plantain. '
            'Preserve the original dish composition, onions, peppers and plate. '
            'Everything blends naturally with identical lighting and photorealistic quality.'
        ),
    },
    {
        'id': 'escargots_grilles',
        'label': 'Escargots africains grillés',
        'mask': 'right_half',
        'prompt': (
            'Professional food photography of an authentic West African alloco dish. '
            'Adding four large grilled African land snails with a glossy seasoned surface, '
            'served beside the fried plantain as a traditional accompaniment. '
            'Keep the original alloco, garnishes and composition unchanged. '
            'Natural restaurant presentation, identical lighting and highly realistic textures.'
        ),
    },
],

    'attieke': [
        {
            'id': 'viande_braisee',
            'label': 'Viande braisée à la place du poisson',
            'mask': 'top_half',
            'prompt': (
                'Food photography of an Ivorian attieke dish. Modifying the top section: replacing the fish with two large '
                'chunks of tender, braised beef coated in a dark, glossy, savory reduction, resting naturally on the '
                'fluffy cassava couscous base with its original onion garnish.'
            ),
        },
        {
            'id': 'poisson_bar',
            'label': 'Bar entier braisé à la place du tilapia',
            'mask': 'top_half',
            'prompt': (
                'Professional food photo of Ivorian attieke. Modifying the fish area: replacing the existing tilapia with '
                'a whole grilled sea bass (bar), glistening skin with perfect grill marks, integrated seamlessly with the '
                'original onion-tomato garnish and attieke base.'
            ),
        },
        {
            'id': 'alloco_cote',
            'label': 'Alloco plantain frit en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography of an Ivorian attieke dish. Modifying the right side of the plate: adding a side portion '
                'of sweet, golden-brown fried ripe plantain slices (alloco) with caramelized edges, seamlessly filling '
                'the empty space while preserving the left side structure.'
            ),
        },
        {
            'id': 'canari_argile',
            'label': 'Servi dans un canari en argile',
            'mask': 'border',
            'prompt': (
                'Traditional plating modification. The existing Ivorian attieke food content is seamlessly transferred '
                'into a rustic, traditional African clay canari pot with a textured reddish-brown terracotta rim, '
                'maintaining original lighting and internal food details.'
            ),
        },
        {
            'id': 'crevettes_braisees',
            'label': 'Grosses crevettes braisées sur l attieke',
            'mask': 'top_half',
            'prompt': (
                'Food photography of Ivorian attieke. Modifying the protein area: replacing the central fish with three '
                'large, succulent braised king prawns, bright pink shells with delicate grill lines, resting beautifully '
                'on the authentic attieke base.'
            ),
        },
    ],

    'kedjenou': [
        {
            'id': 'pintade',
            'label': 'Pintade à la place du poulet',
            'mask': 'center',
            'prompt': (
                'Gourmet food photo of Ivorian kedjenou stew. Modifying the meat inside the vessel: replacing the chicken '
                'pieces with tender, lean chunks of slow-cooked guinea fowl (pintade) shimmering inside the authentic, '
                'rich tomato-pepper broth.'
            ),
        },
        {
            'id': 'attieke_accomp',
            'label': 'Attieke en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography composition. Preserving the existing clay pot of kedjenou stew on the left, and adding '
                'a separate side bowl on the right side containing a clean, fluffy portion of white attieke cassava couscous. '
                'Balanced restaurant plating.'
            ),
        },
        {
            'id': 'canari_argile',
            'label': 'Canari en argile avec couvercle',
            'mask': 'border',
            'prompt': (
                'Authentic restaurant presentation. Modifying the container: the rich chicken kedjenou stew is nested inside '
                'a traditional African earthenware clay canari pot with a textured, unglazed terracotta surface, featuring '
                'a partially open clay lid in the background.'
            ),
        },
        {
            'id': 'poisson_kedjenou',
            'label': 'Poisson entier à la place du poulet',
            'mask': 'center',
            'prompt': (
                'Gourmet food photo of Ivorian kedjenou stew. Modifying the central protein: replacing the chicken chunks '
                'with a whole fresh fish, slow-simmering and beautifully submerged in the rich, vibrant red tomato-onion-pepper '
                'sauce.'
            ),
        },
        {
            'id': 'riz_blanc',
            'label': 'Riz blanc en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography side-by-side composition. Preserving the original kedjenou stew pot on the left, and '
                'adding a separate clean ceramic bowl on the right filled with hot, fluffy steamed white rice grains.'
            ),
        },
    ],

    'gombo': [
        {
            'id': 'crabe_dans_soupe',
            'label': 'Crabe cuit dans la soupe gombo',
            'mask': 'center',
            'prompt': (
                'Close-up food photography of West African okra soup (sauce gombo). Adding a whole cooked crab with '
                'a glossy, vibrant orange-red shell, partially submerged in the thick, dark green viscous okra soup '
                'with visible claws. Harmonious integration.'
            ),
        },
        {
            'id': 'boeuf_kanda',
            'label': 'Boeuf et kanda dans la sauce gombo',
            'mask': 'center',
            'prompt': (
                'Close-up food photography of West African okra soup. Modifying the meat elements: replacing the smoked fish '
                'with chunks of tender braised beef and glistening, gelatinous pieces of cooked beef skin (kanda) '
                'well-integrated into the thick green okra base.'
            ),
        },
        {
            'id': 'foutou_cote',
            'label': 'Foutou banane en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography composition. Preserving the bowl of green okra soup on the left, and cleanly adding '
                'a separate plate on the right side presenting two smooth, pale-golden pounded plantain foutou balls.'
            ),
        },
        {
            'id': 'crevettes_gombo',
            'label': 'Crevettes dans la sauce gombo',
            'mask': 'center',
            'prompt': (
                'Close-up food photography of West African okra soup. Adding three large, plump, pink cooked shrimp '
                'floating naturally on top of the viscous, dark green okra sauce base, blending perfectly with the existing lighting.'
            ),
        },
        {
            'id': 'riz_blanc_gombo',
            'label': 'Riz blanc en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography composition. Keeping the original bowl of okra soup on the left, and introducing '
                'a separate side dish on the right side containing a neat, steaming portion of white rice.'
            ),
        },
    ],

    'thieboudienne': [
    {
        'id': 'yapp_viande',
        'label': 'Thiebu yapp viande à la place du poisson',
        'mask': 'top_half',
        'prompt': (
            'Gourmet photography of Senegalese thieboudienne. Modifying the protein on top: replacing the fish with '
            'succulent, beautifully browned pieces of braised lamb or beef, turning it into the thiebu yapp version '
            'while keeping the savory red tomato rice and vegetables intact.'
        ),
    },
    {
        'id': 'legumes_extra',
        'label': 'Légumes sénégalais supplémentaires',
        'mask': 'bottom_half',
        'prompt': (
            'Gourmet photography of Senegalese thieboudienne. Modifying only the lower portion of the plate (rice and '
            'vegetables area): adding an extra abundance of traditional slow-cooked vegetables, including a tender '
            'carrot, a chunk of cassava root, and a soft cabbage wedge, all glistening with rich tomato broth. '
            'Do not modify or regenerate the fish above — leave it completely untouched.'
        ),
    },
    {
        'id': 'riz_blanc_beurre',
        'label': 'Riz blanc version thiebu beurre',
        'mask': 'bottom_half',
        'prompt': (
            'Gourmet photography of Senegalese thieboudienne. Modifying only the lower portion of the plate: changing '
            'the red tomato rice to a light, fragrant, broken white rice cooked in rich fish broth (thiebu dieun '
            'beurre style). Do not modify or regenerate the fish above — leave it completely untouched.'
        ),
    },
    {
        'id': 'sauce_piment_ramequin',
        'label': 'Petit ramequin de sauce piment à côté',
        'mask': 'bottom_half',
        'prompt': (
            'Gourmet photography of Senegalese thieboudienne. Modifying only the lower/front portion of the frame: '
            'adding a small round side bowl (ramekin) of vibrant green chili sauce placed next to the rice. '
            'Do not modify or regenerate the fish above — leave it completely untouched.'
        ),
    },
    {
        'id': 'fond_table_bois',
        'label': 'Changement du fond : table en bois traditionnelle',
        'mask': 'background',
        'prompt': (
            'Gourmet photography of Senegalese thieboudienne. Modifying only the background/surface behind and '
            'around the plate: replacing it with a warm, rustic wooden table. '
            'Do not modify or regenerate any part of the food — plate, rice, vegetables, and fish must remain '
            'completely untouched and identical.'
        ),
    },
],

    'mafe': [
        {
            'id': 'agneau_braise',
            'label': 'Agneau braisé à la place du boeuf',
            'mask': 'center',
            'prompt': (
                'Professional food photo of West African mafe peanut stew. Modifying the meat: replacing the beef chunks '
                'with highly tender, slow-cooked braised lamb pieces melting into the rich, thick, creamy orange-brown '
                'peanut sauce base.'
            ),
        },
        {
            'id': 'poisson_mafe',
            'label': 'Poisson entier dans la sauce arachide',
            'mask': 'center',
            'prompt': (
                'Professional food photo of West African mafe. Modifying the protein: replacing the meat with a whole '
                'fresh fish, gracefully simmering and partially submerged in the rich, thick savory peanut sauce.'
            ),
        },
        {
            'id': 'fonio_accomp',
            'label': 'Fonio en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography presentation. Preserving the creamy mafe peanut stew on the left, and seamlessly '
                'adding a small side bowl on the right containing a fluffy portion of steamed golden fonio grains.'
            ),
        },
        {
            'id': 'legumes_racines',
            'label': 'Légumes racines dans la sauce arachide',
            'mask': 'center',
            'prompt': (
                'Professional food photo of West African mafe peanut stew. Modifying the stew contents: adding tender, '
                'chunks of sweet potato and soft sliced carrots, their warm orange colors vibrant against the thick peanut sauce.'
            ),
        },
        {
            'id': 'riz_accomp',
            'label': 'Riz blanc en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography presentation. Keeping the original mafe peanut stew bowl on the left, and neatly '
                'adding a side bowl of fluffy, hot steamed white rice on the right side.'
            ),
        },
    ],

    'yassa': [
        {
            'id': 'poisson_yassa',
            'label': 'Poisson grillé à la place du poulet',
            'mask': 'top_half',
            'prompt': (
                'Gourmet photo of Senegalese yassa. Modifying the central protein: replacing the chicken with one whole '
                'perfectly grilled tilapia fish, nestled beautifully beneath the generous blanket of sweet, caramelized '
                'onion and mustard-lemon sauce over white rice.'
            ),
        },
        {
            'id': 'agneau_yassa',
            'label': 'Agneau à la place du poulet',
            'mask': 'top_half',
            'prompt': (
                'Gourmet photo of Senegalese yassa. Modifying the protein: replacing the chicken with meltingly tender '
                'pieces of braised lamb chunks, well-incorporated under the rich caramelized onion and zesty mustard-lemon sauce.'
            ),
        },
        {
            'id': 'oignons_genereux',
            'label': 'Portion d oignons caramélisés plus généreux',
            'mask': 'top_half',
            'prompt': (
                'Gourmet photo of Senegalese yassa poulet. Modifying the top layer: multiplying the amount of caramelized '
                'golden onions, creating an incredibly rich, glistening, and generous heap of mustard-lemon onion sauce '
                'cascading over the chicken and rice.'
            ),
        },
        {
            'id': 'canari_yassa',
            'label': 'Servi dans un canari traditionnel',
            'mask': 'border',
            'prompt': (
                'Traditional restaurant presentation. Modifying the container: the authentic chicken yassa and rice are '
                'elegantly nested inside a traditional African clay canari pot made of warm, textured reddish-brown terracotta.'
            ),
        },
        {
            'id': 'crevettes_yassa',
            'label': 'Crevettes à la place du poulet',
            'mask': 'top_half',
            'prompt': (
                'Gourmet photo of Senegalese yassa. Modifying the protein: replacing the chicken with a luxurious serving '
                'of large grilled prawns, pink and lightly charred, layered under the zesty caramelized onion sauce.'
            ),
        },
    ],

    'thiakry': [
        {
            'id': 'glacons_dessus',
            'label': 'Glaçons sur le thiakry froid',
            'mask': 'top_half',
            'prompt': (
                'Macro food photography of West African thiakry dessert. Adding five clear, realistic ice cubes resting '
                'on top of the creamy millet-and-yogurt base, complete with tiny melting water droplets for a highly '
                'chilled, fresh presentation.'
            ),
        },
        {
            'id': 'mangue_topping',
            'label': 'Topping mangue fraîche tranchée',
            'mask': 'top_half',
            'prompt': (
                'Macro food photography of West African thiakry. Adding vibrant, juicy slices of fresh ripe mango beautifully '
                'fanned out on top of the smooth, white millet-yogurt cream dessert base. High contrast.'
            ),
        },
        {
            'id': 'coco_raisins',
            'label': 'Noix de coco râpée et raisins secs dessus',
            'mask': 'top_half',
            'prompt': (
                'Macro food photography of West African thiakry. Adding a delicate sprinkle of fine shredded coconut flakes '
                'and plump golden raisins scattered elegantly across the surface of the creamy millet dessert.'
            ),
        },
        {
            'id': 'calebasse_thiakry',
            'label': 'Servi dans une calebasse',
            'mask': 'border',
            'prompt': (
                'Traditional styling. Modifying the container: the sweet millet-yogurt thiakry dessert is seamlessly presented '
                'inside a rustic, organic African calabash gourd bowl with a smooth, natural beige-brown finish, replacing the original rim.'
            ),
        },
        {
            'id': 'fruits_tropicaux',
            'label': 'Fruits tropicaux en topping mangue banane',
            'mask': 'top_half',
            'prompt': (
                'Macro food photography of West African thiakry. Adding a colorful, gourmet arrangement of fresh tropical '
                'fruits on top: sliced ripe mango, neat banana rounds, and a vibrant red strawberry garnish.'
            ),
        },
    ],

    'domoda': [
        {
            'id': 'poisson_domoda',
            'label': 'Poisson à la place de la viande',
            'mask': 'center',
            'prompt': (
                'Professional food shot of Gambian domoda stew. Modifying the main protein: replacing the meat with '
                'a whole fish, simmered carefully and emerging slightly from the thick, savory peanut-tomato sauce base.'
            ),
        },
        {
            'id': 'courge_extra',
            'label': 'Plus de courge et patate douce',
            'mask': 'center',
            'prompt': (
                'Professional food shot of Gambian domoda stew. Modifying the vegetable density: adding extra generous '
                'chunks of bright orange butternut squash and tender sweet potato, glistening within the rich peanut-tomato gravy.'
            ),
        },
        {
            'id': 'agneau_domoda',
            'label': 'Agneau à la place du boeuf',
            'mask': 'center',
            'prompt': (
                'Professional food shot of Gambian domoda stew. Modifying the protein: replacing the beef with tender '
                'chunks of slow-cooked lamb, beautifully integrated into the deep orange-brown peanut-tomato sauce.'
            ),
        },
        {
            'id': 'riz_accomp',
            'label': 'Riz blanc en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photography layout. Keeping the rich domoda peanut stew bowl completely unchanged on the left, '
                'and neatly introducing a separate white ceramic bowl filled with fluffy white rice on the right side.'
            ),
        },
        {
            'id': 'argile_contenant',
            'label': 'Servi dans un bol en argile',
            'mask': 'border',
            'prompt': (
                'Traditional styling modification. Modifying the vessel: the rich Gambian domoda stew is presented inside '
                'a beautiful, handmade African clay earthenware bowl with an authentic, textured reddish-brown terracotta rim.'
            ),
        },
    ],

    'generique': [
        {
            'id': 'viande_braisee',
            'label': 'Viande braisée ajoutée',
            'mask': 'center',
            'prompt': (
                'Professional food photo of an African dish. Modifying the center: adding savory, tender chunks of braised '
                'beef glistening with a rich dark reduction sauce, seamlessly blended into the existing food arrangement.'
            ),
        },
        {
            'id': 'calebasse',
            'label': 'Servi dans une calebasse africaine',
            'mask': 'border',
            'prompt': (
                'Traditional plating change. Modifying the container: the existing African food is presented inside an '
                'authentic, rustic traditional African calabash gourd bowl, replacing the original plate border naturally.'
            ),
        },
        {
            'id': 'riz_accomp',
            'label': 'Riz blanc en accompagnement',
            'mask': 'right_half',
            'prompt': (
                'Food photo layout. Preserving the original African dish on the left side of the frame, and adding '
                'a separate small side dish containing a neat portion of steaming white rice on the right.'
            ),
        },
        {
            'id': 'legumes_extra',
            'label': 'Légumes supplémentaires',
            'mask': 'bottom_half',
            'prompt': (
                'Professional food photo of an African dish. Modifying the lower section: adding a clean, appetizing arrangement '
                'of slow-cooked root vegetables like carrots, sweet potatoes, or cassava, perfectly integrated into the sauce.'
            ),
        },
        {
            'id': 'canari_argile',
            'label': 'Contenant canari en argile',
            'mask': 'border',
            'prompt': (
                'Traditional plating change. Modifying the container: the authentic African food is now nested inside '
                'a traditional, rustic clay canari pot with a distinct reddish-brown terracotta texture, replacing the original plate edge.'
            ),
        },
    ],
}

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

print(f'Chargement {SD_MODEL}...')
print('Premier lancement : ~1.7GB (2-3 min)...')

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    SD_MODEL,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to('cuda')

try:
    pipe.enable_xformers_memory_efficient_attention()
    print('✅ xFormers activé')
except: pass
pipe.enable_model_cpu_offload()
print('✅ Modèle SD prêt !')

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Chargement stable-diffusion-v1-5/stable-diffusion-inpainting...
Premier lancement : ~1.7GB (2-3 min)...


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


✅ xFormers activé
✅ Modèle SD prêt !


In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def find_image(folder, image_id):
    folder = Path(folder)
    for ext in ['', '.jpg', '.jpeg', '.png', '.webp', '.JPG']:
        p = folder / f'{image_id}{ext}'
        if p.exists(): return p
    stem = Path(image_id).stem.lower()
    for f in folder.rglob('*'):
        if f.is_file() and f.stem.lower() == stem: return f
    return None

def make_mask(size, zone):
    w, h = size
    mask = Image.new('L', size, 0)
    d    = ImageDraw.Draw(mask)
    m    = int(min(w,h)*0.15)
    zones = {
        'full':        [(0,0,w,h)],
        'border':      [(0,0,w,m),(0,h-m,w,h),(0,0,m,h),(w-m,0,w,h)],
        'center':      [(m,m,w-m,h-m)],
        'top_half':    [(0,0,w,h//2)],
        'bottom_half': [(0,h//2,w,h)],
        'right_half':  [(w//2,0,w,h)],
        'top_right':   [(w//2,0,w,h//2)],
    }
    for rect in zones.get(zone, [(0,0,w,h)]):
        d.rectangle(rect, fill=255)
    return mask.filter(ImageFilter.GaussianBlur(14))

NEG = (
    'blurry, deformed, distorted, ugly, cartoon, text, watermark, '
    'duplicate plate, wrong food, extra dish, unrealistic'
)

def sd_inpaint(img, zone, prompt):
    mask = make_mask(img.size, zone)
    return pipe(
        prompt=prompt,
        negative_prompt=NEG,
        image=img,
        mask_image=mask,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE,
        strength=STRENGTH,
        width=IMG_SIZE, height=IMG_SIZE,
    ).images[0]

def exporter_excel(df_orig, df_new, chemin):
    combined = pd.concat([df_orig, df_new], ignore_index=True)
    with pd.ExcelWriter(chemin, engine='openpyxl') as writer:
        df_new.to_excel(writer,   sheet_name='Images_Generees',    index=False)
        combined.to_excel(writer, sheet_name='Toutes_Annotations', index=False)
        if 'gen_type' in df_new.columns:
            r = df_new.groupby(['gen_type', 'gen_detail']).size().reset_index(name='count')
            r.to_excel(writer, sheet_name='Resume', index=False)
        COLORS = {'Images_Generees':'1B4332','Toutes_Annotations':'2D6A4F','Resume':'40916C'}
        for sn, ws in writer.sheets.items():
            c = COLORS.get(sn, '2D6A4F')
            for cell in ws[1]:
                cell.fill      = PatternFill('solid', fgColor=c)
                cell.font      = Font(bold=True, color='FFFFFF', size=11)
                cell.alignment = Alignment(horizontal='center')
            for i, col in enumerate(ws.columns, 1):
                ml = max((len(str(x.value)) if x.value else 0) for x in col)
                ws.column_dimensions[get_column_letter(i)].width = min(ml+4, 65)
            ws.freeze_panes    = 'A2'
            ws.auto_filter.ref = ws.dimensions

print('✅ Utilitaires prêts')

✅ Utilitaires prêts


In [ ]:
# ════════════════════════════════════════════════════════════════
# 🚀  GÉNÉRATION PRINCIPALE
#
# Pour chaque image originale :
#   → 5 SD  : modifications sémantiques du contenu du plat
#   → 5 PIL : variations d'environnement/éclairage
#
# Chaque image générée hérite de TOUTES les colonnes originales.
# 4 colonnes ajoutées : gen_image_id, gen_source, gen_type, gen_detail
# ════════════════════════════════════════════════════════════════

df = pd.read_excel(EXCEL_INPUT, sheet_name=SHEET_INDEX)
print(f'📊 {len(df)} images originales → {len(df)*10} à générer')

if IMAGE_COL not in df.columns:
    print(f'❌ Colonne "{IMAGE_COL}" introuvable. Colonnes : {list(df.columns)}')
else:
    print('\n🔍 Détection des plats (aperçu) :')
    for _, row in df.head(5).iterrows():
        img_id = str(row[IMAGE_COL])
        print(f'   {img_id:35s} → {detecter_plat(img_id)}')
    print()

    new_rows=[]; errors=[]; tot_sd=0; tot_pil=0

    for idx, row in df.iterrows():
        image_id = str(row[IMAGE_COL]).strip()
        img_path = find_image(IMAGES_FOLDER, image_id)

        if img_path is None:
            print(f'⚠️  Introuvable : {image_id}'); errors.append(image_id); continue

        plat   = detecter_plat(image_id)
        ajouts = SD_AJOUTS.get(plat, SD_AJOUTS['generique'])
        stem   = Path(image_id).stem

        print(f'\n{"─"*62}')
        print(f'  [{idx+1}/{len(df)}]  {image_id}  →  {plat}')
        print(f'{"─"*62}')

        base = Image.open(img_path).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.LANCZOS)

        # ── 5 modifications SD ─────────────────────────────────────
        print('  🤖 SD — modifications du contenu...')
        for ajout in ajouts[:N_SD]:
            print(f'    {ajout["label"]} ... ', end='', flush=True)
            try:
                out    = sd_inpaint(base, ajout['mask'], ajout['prompt'])
                new_id = f'{stem}_sd_{ajout["id"]}.jpg'
                out.save(Path(OUTPUT_FOLDER)/new_id, quality=95)
                nr = row.to_dict()
                nr[IMAGE_COL]      = new_id
                nr['gen_image_id'] = new_id
                nr['gen_source']   = image_id
                nr['gen_type']     = 'SD_contenu'
                nr['gen_detail']   = ajout['label']
                new_rows.append(nr); tot_sd+=1
                print('✅')
            except Exception as e:
                print(f'❌  {str(e)[:60]}'); errors.append(f'{image_id}_sd_{ajout["id"]}')

        # ── 5 transformations PIL ─────────────────────────────────
        print('  📐 PIL — variations environnement...')
        for nom, fn in PIL_RECETTES[:N_PIL]:
            try:
                out    = fn(base).convert('RGB')
                new_id = f'{stem}_pil_{nom}.jpg'
                out.save(Path(OUTPUT_FOLDER)/new_id, quality=95)
                nr = row.to_dict()
                nr[IMAGE_COL]      = new_id
                nr['gen_image_id'] = new_id
                nr['gen_source']   = image_id
                nr['gen_type']     = 'PIL_environnement'
                nr['gen_detail']   = nom
                new_rows.append(nr); tot_pil+=1
                print(f'    ✅  {nom}')
            except Exception as e:
                print(f'    ❌  {nom} — {str(e)[:50]}'); errors.append(f'{image_id}_pil_{nom}')

        # Checkpoint toutes les 50 images originales
        if (idx+1)%50==0 and new_rows:
            tmp = pd.DataFrame(new_rows)
            exporter_excel(df, tmp, EXCEL_OUTPUT.replace('.xlsx','_checkpoint.xlsx'))
            print(f'\n  💾 Checkpoint : {tot_sd+tot_pil} images\n')

    print(f'\n{"═"*62}')
    print(f'  ✅  SD  : {tot_sd}')
    print(f'  ✅  PIL : {tot_pil}')
    print(f'  ✅  TOTAL : {tot_sd+tot_pil}  →  {OUTPUT_FOLDER}')
    if errors: print(f'  ⚠️   Erreurs : {len(errors)}')
    print(f'{"═"*62}')




📊 4 images originales → 40 à générer

🔍 Détection des plats (aperçu) :
   Kedjenou_001.jpg                    → kedjenou
   Kedjenou_002.jpg                    → kedjenou
   Kedjenou_003.jpg                    → kedjenou
   foutou_004.jpg                      → kedjenou


──────────────────────────────────────────────────────────────
  [1/4]  Kedjenou_001.jpg  →  kedjenou
──────────────────────────────────────────────────────────────
  🤖 SD — modifications du contenu...
    Pintade à la place du poulet ... 

  0%|          | 0/30 [00:00<?, ?it/s]

✅
    Attieke en accompagnement ... 

  0%|          | 0/30 [00:00<?, ?it/s]

✅
    Canari en argile avec couvercle ... 

...

✅
    Poisson entier à la place du poulet ... 

  0%|          | 0/30 [00:00<?, ?it/s]

✅
    Riz blanc en accompagnement ... 

  0%|          | 0/30 [00:00<?, ?it/s]

✅
  📐 PIL — variations environnement...
    ✅  env_lumiere_chaude
    ✅  env_flash_contraste
    ✅  env_exterieur_froid
    ✅  env_miroir_lateral
    ✅  env_nuit_maquis

══════════════════════════════════════════════════════════════
  ✅  SD  : 20
  ✅  PIL : 20
  ✅  TOTAL : 40  →  images_augmentees/
══════════════════════════════════════════════════════════════


In [ ]:
if new_rows:
    df_new = pd.DataFrame(new_rows)
    exporter_excel(df, df_new, EXCEL_OUTPUT)
    print(f'✅ Excel exporté : {EXCEL_OUTPUT}')
    print(f'   Images_Generees    : {len(df_new)} nouvelles images')
    print(f'   Toutes_Annotations : {len(df)+len(df_new)} au total')
    print(f'   Colonnes ajoutees  : gen_image_id, gen_source, gen_type, gen_detail')
else:
    print('⚠️  Aucune image générée')

✅ Excel exporté : annotations_augmentees.xlsx
   Images_Generees    : 1000 nouvelles images
   Toutes_Annotations : 1100 au total
   Colonnes ajoutees  : gen_image_id, gen_source, gen_type, gen_detail


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
sd_imgs  = sorted(_glob.glob(f'{OUTPUT_FOLDER}*_sd_*.jpg'))[:5]
pil_imgs = sorted(_glob.glob(f'{OUTPUT_FOLDER}*_pil_*.jpg'))[:5]

if sd_imgs or pil_imgs:
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    for ax, p in zip(axes[0], sd_imgs):
        ax.imshow(Image.open(p))
        ax.set_title(f'[SD] {Path(p).stem[-22:]}', fontsize=7, color='#1B4332')
        ax.axis('off')
    for ax in axes[0][len(sd_imgs):]: ax.axis('off')
    for ax, p in zip(axes[1], pil_imgs):
        ax.imshow(Image.open(p))
        ax.set_title(f'[PIL] {Path(p).stem[-22:]}', fontsize=7, color='#2D6A4F')
        ax.axis('off')
    for ax in axes[1][len(pil_imgs):]: ax.axis('off')
    plt.suptitle(
        'Ligne 1 : SD (modifications contenu)  |  Ligne 2 : PIL (variations environnement)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('apercu.png', dpi=120, bbox_inches='tight')
    plt.show()
    total = len(_glob.glob(f'{OUTPUT_FOLDER}*.jpg'))
    print(f'Total : {total} images  (SD:{len(_glob.glob(f"{OUTPUT_FOLDER}*_sd_*.jpg"))} + PIL:{len(_glob.glob(f"{OUTPUT_FOLDER}*_pil_*.jpg"))})')

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import zipfile  # <--- Ajout indispensable pour corriger ton erreur !
import glob
with zipfile.ZipFile('foutou01.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for img in _glob.glob(f'{OUTPUT_FOLDER}*.jpg'): zf.write(img)
    if os.path.exists(EXCEL_OUTPUT): zf.write(EXCEL_OUTPUT)
    if os.path.exists('apercu.png'):  zf.write('apercu.png')

print('✅ ZIP prêt — téléchargement...')
files.download('foutou01.zip')
files.download(EXCEL_OUTPUT)

✅ ZIP prêt — téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>